# Austrian Energy Projections

This notebook processes the energy projections as published by the Umweltbundesamt for use in the energy-scenarios-at Scenario Explorer.

## Latest Projections Update (2025)

### Report

https://www.umweltbundesamt.at/studien-reports/publikationsdetail?pub_id=2616

> Energie- und Treibhausgasszenarien 2025.  
> Wien, 2025  
> Reports, Band 0995  
> ISBN: 987-3-99004-842-9

Data tables were extracted from the pdf report using AI.

Permission to republish under a CC-BY license by email on March 18, 2026.

In [ ]:
import nomenclature
import pyam

In [ ]:
from uba_utils import read_uba_file

In [ ]:
df_args = dict(
    model="Umweltbundesamt (2025)",
    region="Austria",
)

In [ ]:
file = "source/rep0995_tables.xlsx"

In [ ]:
definition = nomenclature.DataStructureDefinition("../../definitions/")

## Final energy consumption

In [ ]:
prefix = "Final Energy [by Sector]|"

energy_by_sector_mapping = {
    "Verkehr": prefix + "Transportation",
    "Industrie": prefix + "Industry",
    "Haushalte": prefix + "Residential and Commercial|Residential",
    "Dienstleistungen":
      prefix + "Residential and Commercial|Commercial and Institutional",
    "Landwirtschaft": prefix + "Agriculture",
    "EEV": "Final Energy",
}

In [ ]:
df_final_energy_by_sector = pyam.concat(
    [
        read_uba_file(
            file=file,
            sheet_name="Tabelle 10",
            skiprows=3,
            nrows=6,
            variable_col="Sektoren",
            unit="PJ",
            scenario=scenario,
            usecols=usecols,
            col_suffix=col_suffix,
            variable_mapping=energy_by_sector_mapping,
            df_args=df_args,
        )
        for scenario, usecols, col_suffix in (
            ("With Existing Measures (WEM 2025)", "A:E", None),
            ("With Additional Measures (WAM 2025)", "A:B,F:H", ".1"),
        )
    ]
)

In [ ]:
energy_by_fuel_mapping = {
    "Kohle": "Final Energy [by Carrier]|Coal",
    "Öl": "Final Energy [by Carrier]|Oil",
    "Erdgas": "Final Energy [by Carrier]|Natural Gas",
    "Biomasse (inkl. Biomethan)": "Final Energy [by Carrier]|Biomass",
    "Abfall": "Final Energy [by Carrier]|Waste",
    "Wasserstoff; e-Fuels": "Final Energy [by Carrier]|Hydrogen and E-Fuels",
    "Strom": "Final Energy [by Carrier]|Electricity",
    "Umgebungswärme etc.*": "Final Energy [by Carrier]|Ambient Heat",
    "Fernwärme": "Final Energy [by Carrier]|District Heat",
}

In [ ]:
df_final_energy_by_carrier = pyam.concat(
    [
        read_uba_file(
            file=file,
            sheet_name="Tabelle 12",
            skiprows=3,
            nrows=10,
            variable_col="Energieträger",
            unit="PJ",
            scenario=scenario,
            usecols=usecols,
            col_suffix=col_suffix,
            variable_mapping=energy_by_fuel_mapping,
            df_args=df_args,
        )
        for scenario, usecols, col_suffix in (
            ("With Existing Measures (WEM 2025)", "A:E", None),
            ("With Additional Measures (WAM 2025)", "A:B,F:H", ".1"),
        )
    ]
)

In [ ]:
df_final_energy_by_sector.aggregate(
    "Final Energy [by Sector]|Residential and Commercial",
    append=True,
)

In [ ]:
df_energy = pyam.concat(
    [
        df_final_energy_by_sector,
        df_final_energy_by_carrier,
    ]
).convert_unit("PJ", "TJ")

In [ ]:
df_energy.check_aggregate(
    "Final Energy",
    components=df_energy.filter(variable="Final Energy [by Carrier]*").variable,
    rtol=0.05,  # aggregates do not match precisely due to roudning errors
)

## Final energy consumption by sectors

In [ ]:
prefix = "Final Energy [by Sector]|Residential and Commercial|"

rescom_energy_by_carrier_mapping = {
    "Öl": prefix + "Oil",
    "Gas": prefix + "Natural Gas",
    "Biomasse (inkl. Biomethan)": prefix + "Biomass",
    "Strom": prefix + "Electricity",
    "Wärme*": prefix + "District Heat and Ambient Heat",
}

In [ ]:
df_final_energy_res_com = pyam.concat(
    [
        read_uba_file(
            file=file,
            sheet_name="Tabelle 14",
            skiprows=3,
            nrows=6,
            variable_col="Energieträger",
            unit="PJ",
            scenario=scenario,
            usecols=usecols,
            col_suffix=col_suffix,
            variable_mapping=rescom_energy_by_carrier_mapping,
            df_args=df_args,
        )
        for scenario, usecols, col_suffix in (
            ("With Existing Measures (WEM 2025)", "A:E", None),
            ("With Additional Measures (WAM 2025)", "A:B,F:H", ".1"),
        )
    ]
).convert_unit("PJ", "TJ")

In [ ]:
prefix = "Final Energy [by Sector]|Transportation|"

transpiort_energy_by_carrier_mapping = {
    "Öl": prefix + "Oil",
    "Erdgas": prefix + "Natural Gas",
    "Biomasse (inkl. Biomethan)": prefix + "Biomass",
    "Wasserstoff; e-Fuels": prefix + "Hydrogen and E-Fuels",
    "Strom": prefix + "Electricity",
}

In [ ]:
df_final_energy_transportation = pyam.concat(
    [
        read_uba_file(
            file=file,
            sheet_name="Tabelle 15",
            skiprows=3,
            nrows=6,
            variable_col="Energieträger",
            unit="PJ",
            scenario=scenario,
            usecols=usecols,
            col_suffix=col_suffix,
            variable_mapping=transpiort_energy_by_carrier_mapping,
            df_args=df_args,
        )
        for scenario, usecols, col_suffix in (
            ("With Existing Measures (WEM 2025)", "A:E", None),
            ("With Additional Measures (WAM 2025)", "A:B,F:H", ".1"),
        )
    ]
).convert_unit("PJ", "TJ")

In [ ]:
prefix = "Final Energy [by Sector]|Agriculture|"

agriculture_energy_by_carrier_mapping = {
    "Öl": prefix + "Oil",
    "Erdgas": prefix + "Natural Gas",
    "Biomasse (inkl. Biomethan)": prefix + "Biomass",
    "Strom": prefix + "Electricity",
    "Wärme": prefix + "District Heat and Ambient Heat",
}

In [ ]:
df_final_energy_agriculture = pyam.concat(
    [
        read_uba_file(
            file=file,
            sheet_name="Tabelle 16",
            skiprows=3,
            nrows=5,
            variable_col="Energieträger",
            unit="PJ",
            scenario=scenario,
            usecols=usecols,
            col_suffix=col_suffix,
            variable_mapping=agriculture_energy_by_carrier_mapping,
            df_args=df_args,
        )
        for scenario, usecols, col_suffix in (
            ("With Existing Measures (WEM 2025)", "A:E", None),
            ("With Additional Measures (WAM 2025)", "A:B,F:H", ".1"),
        )
    ]
).convert_unit("PJ", "TJ")

In [ ]:
prefix = "Final Energy [by Sector]|Industry|"

industry_energy_by_carrier_mapping = {
    "Kohle": prefix + "Coal",
    "Öl": prefix + "Oil",
    "Erdgas": prefix + "Natural Gas",
    "Biomasse (inkl. Biomethan)": prefix + "Biomass",
    "Waste": prefix + "Waste",
    "Strom": prefix + "Electricity",
    "Wärme": prefix + "District Heat and Ambient Heat",
}

In [ ]:
df_final_energy_industry = pyam.concat(
    [
        read_uba_file(
            file=file,
            sheet_name="Tabelle 17",
            skiprows=3,
            nrows=8,
            variable_col="Energieträger",
            unit="PJ",
            scenario=scenario,
            usecols=usecols,
            col_suffix=col_suffix,
            variable_mapping=industry_energy_by_carrier_mapping,
            df_args=df_args,
        )
        for scenario, usecols, col_suffix in (
            ("With Existing Measures (WEM 2025)", "A:E", None),
            ("With Additional Measures (WAM 2025)", "A:B,F:H", ".1"),
        )
    ]
).convert_unit("PJ", "TJ")

## Power generation

In [ ]:
electricity_by_source_mapping = {
    "fossil": "Secondary Energy|Electricity|Natural Gas",
    "Wasserkraft": "Secondary Energy|Electricity|Hydro",
    "Biomasse (inkl. Biomethan)": "Secondary Energy|Electricity|Biomass",
    "Umgebungswärme etc.*": "Secondary Energy|Electricity|Geothermal",
    "Photovoltaik": "Secondary Energy|Electricity|Solar",
    "Wind": "Secondary Energy|Electricity|Wind",
#    "Grüner Wasserstoff": None, # this is all zero in the data file
    "Stromerzeugung": "Secondary Energy|Electricity",
    "Nettoimporte": "Net Imports|Electricity",
}

In [ ]:
df_electricity_by_source = pyam.concat(
    [
        read_uba_file(
            file=file,
            sheet_name="Tabelle 20",
            skiprows=3,
            nrows=9,
            variable_col="Energieträger",
            unit="TWh",
            scenario=scenario,
            usecols=usecols,
            col_suffix=col_suffix,
            variable_mapping=electricity_by_source_mapping,
            df_args=df_args,
        )
        for scenario, usecols, col_suffix in (
            ("With Existing Measures (WEM 2025)", "A:E", None),
            ("With Additional Measures (WAM 2025)", "A:B,F:H", ".1"),
        )
    ]
)

In [ ]:
df_electricity_by_source.check_aggregate(
    "Secondary Energy|Electricity", rtol=0.05
)

## Concatenate, validate, export

In [ ]:
df = pyam.concat(
    [
        df_energy,
        df_electricity_by_source,
        df_final_energy_res_com,
        df_final_energy_transportation,
        df_final_energy_agriculture,
        df_final_energy_industry,
    ]
)

In [ ]:
for sector in ["Agriculture", "Industry", "Transportation"]:
    df.check_aggregate(f"Final Energy [by Sector]|{sector}", rtol=0.1)

In [ ]:
prefix = "Final Energy [by Sector]|Residential and Commercial|"
(
    df
    .filter(
        variable=[
            prefix + "Residential",
            prefix + "Commercial and Institutional",
        ],
        keep=False,
    )
    .check_aggregate(
        "Final Energy [by Sector]|Residential and Commercial",
        rtol=0.1
    )
)

In [ ]:
definition.validate(df)

In [ ]:
df.to_excel("uba_wem_wam_2025_energy.xlsx")